# Section 1: Timestamp Standardization (Europe/Berlin, 24-Hour Format)

## Objective

Load the saved ICE dataset, convert all five timestamp columns to a consistent `Europe/Berlin` timezone with 24-hour datetime format, create the merge key `departure_planned_hour`, and save the result to disk.

## Why this step is important

Railway and weather data must share the same timezone and hour precision before merging. Mixed or naive timestamps cause silent join failures.

## What problem does it solve?

It answers: *Are all railway timestamps in one consistent format ready for the weather merge?*

## Methodology

1. Load `ice_raw_2024-07.parquet` from disk (not Hugging Face).
2. Parse 5 timestamp columns with `pd.to_datetime`.
3. Localize naive timestamps to `Europe/Berlin` (German rail local time).
4. Create `departure_planned_hour` = floor of `departure_planned_time` to the hour.
5. Validate: no parsing failures, plausible date range, hour key created.
6. Save `ice_standardized_2024-07.parquet` + report JSON.

**Not done yet:** cancellation removal and null handling → Section 2.

## Expected Output

- Before/after dtype comparison for timestamp columns
- Null count in timestamp columns after parsing
- Sample rows showing raw time vs `departure_planned_hour`
- Saved standardized parquet file

## Interpretation

- **dtype `datetime64[ns, Europe/Berlin]`** → timezone-aware, correct
- **NaT values** → originally null timestamps (e.g., first stop has no arrival)
- **`departure_planned_hour`** → `14:37` becomes `14:00` — matches Open-Meteo hourly data
- **Row count unchanged** → standardization only, no rows dropped yet

## Challenges Faced

| Challenge | Approach |
|-----------|----------|
| Naive vs UTC timestamps | Localize to `Europe/Berlin` explicitly |
| DST ambiguous times | `ambiguous='infer'` with NaT logging |
| Null departure times | Keep as NaT; merge handled in Notebook 04 |

## Summary

All timestamps standardized to `Europe/Berlin` 24-hour format. Merge key `departure_planned_hour` created and saved.

## Next Step

**Section 2:** Remove canceled stops, handle nulls, validate cleaned dataset → save `ice_cleaned_2024-07.parquet`.

---

**Key Takeaways**
- 5 timestamp columns standardized; merge key created
- Output: `ice_standardized_2024-07.parquet`
- Row count same as input — no filtering yet

**What Comes Next**
- Section 2: exclusion rules and final cleaned dataset

In [1]:
# =============================================================================
# Notebook 03 | Section 1: Timestamp Standardization (Europe/Berlin)
# =============================================================================
# Self-contained: loads ICE raw parquet + configs from disk.
# Saves ice_standardized_2024-07.parquet for Section 2.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# =============================================================================
# 1. PATHS
# =============================================================================
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CONFIG_PATH = REFERENCE_DIR / "project_config.json"
MERGE_STRATEGY_PATH = REFERENCE_DIR / "merge_strategy.json"
NB02_SUMMARY_PATH = REFERENCE_DIR / "notebook_02_summary.json"

TARGET_MONTH = "2024-07"
INPUT_PARQUET = PROCESSED_DIR / f"ice_raw_{TARGET_MONTH}.parquet"
OUTPUT_PARQUET = PROCESSED_DIR / f"ice_standardized_{TARGET_MONTH}.parquet"
STANDARDIZE_REPORT_PATH = REFERENCE_DIR / f"timestamp_standardize_report_{TARGET_MONTH}.json"

TARGET_TZ = "Europe/Berlin"

TIMESTAMP_COLS = [
    "time",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


# =============================================================================
# 2. LOAD INPUT DATA + CONFIG
# =============================================================================
config = load_json(CONFIG_PATH)
merge_strategy = load_json(MERGE_STRATEGY_PATH)
VERIFIED_COLUMNS = [c["name"] for c in config["db_verified_columns"]]

if not INPUT_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing: {INPUT_PARQUET}\nRun Notebook 02 Section 1 first."
    )

print(f"Loading: {INPUT_PARQUET}")
ice_df = pd.read_parquet(INPUT_PARQUET)
rows_before = len(ice_df)
print(f"Rows loaded: {rows_before:,}")
print()

# Record dtypes BEFORE standardization
dtypes_before = {col: str(ice_df[col].dtype) for col in TIMESTAMP_COLS}

# =============================================================================
# 3. TIMESTAMP STANDARDIZATION FUNCTION
# =============================================================================
def standardize_to_berlin(
    series: pd.Series,
    tz: str = TARGET_TZ,
) -> tuple[pd.Series, int]:
    """
    Convert a datetime series to timezone-aware Europe/Berlin.

    Steps:
      1. Parse strings/objects to datetime (NaT for unparseable)
      2. If naive → localize to Europe/Berlin
      3. If already tz-aware → convert to Europe/Berlin

    Returns
    -------
    standardized_series, nat_count
    """
    parsed = pd.to_datetime(series, errors="coerce")
    nat_before = int(parsed.isna().sum())

    if parsed.dt.tz is None:
        # Naive timestamps: treat as local German rail time
        localized = parsed.dt.tz_localize(
            tz,
            ambiguous="infer",       # infer during DST fall-back
            nonexistent="shift_forward",  # handle spring-forward gap
        )
    else:
        localized = parsed.dt.tz_convert(tz)

    nat_after = int(localized.isna().sum())
    new_nats = nat_after - nat_before  # Nats introduced by DST issues
    return localized, new_nats


# =============================================================================
# 4. APPLY STANDARDIZATION TO ALL 5 TIMESTAMP COLUMNS
# =============================================================================
dst_issues = {}

for col in TIMESTAMP_COLS:
    ice_df[col], new_nats = standardize_to_berlin(ice_df[col])
    if new_nats > 0:
        dst_issues[col] = new_nats
    print(
        f"  {col:<28} → {ice_df[col].dtype}  "
        f"(null: {ice_df[col].isna().sum():,})"
    )

print()

# =============================================================================
# 5. CREATE MERGE KEY: departure_planned_hour
# =============================================================================
# Floor to hour in Europe/Berlin — matches Open-Meteo hourly.time
ice_df["departure_planned_hour"] = (
    ice_df["departure_planned_time"]
    .dt.floor("h")
)

# Store as timezone-naive for easier join with weather time column
# (weather time from API is naive Europe/Berlin)
ice_df["departure_planned_hour_naive"] = (
    ice_df["departure_planned_hour"]
    .dt.tz_localize(None)
)

merge_key_nulls = int(ice_df["departure_planned_hour"].isna().sum())
merge_key_null_pct = round(100 * merge_key_nulls / len(ice_df), 2)

print(f"Merge key created: departure_planned_hour")
print(f"  Null merge keys : {merge_key_nulls:,}  ({merge_key_null_pct}%)")
print(f"  Unique hours    : {ice_df['departure_planned_hour_naive'].nunique():,}")
print()

# =============================================================================
# 6. VALIDATION CHECKS
# =============================================================================
validations = {}

# Row count must not change (no filtering in this section)
validations["row_count_unchanged"] = {
    "pass": len(ice_df) == rows_before,
    "before": rows_before,
    "after": len(ice_df),
}

# All timestamp cols must be tz-aware Europe/Berlin
for col in TIMESTAMP_COLS:
    is_berlin = str(ice_df[col].dtype) == "datetime64[ns, Europe/Berlin]"
    validations[f"tz_aware_{col}"] = {"pass": is_berlin, "dtype": str(ice_df[col].dtype)}

# Date range sanity check
validations["date_range"] = {
    "time_min": str(ice_df["time"].min()),
    "time_max": str(ice_df["time"].max()),
    "departure_planned_min": str(ice_df["departure_planned_time"].min()),
    "departure_planned_max": str(ice_df["departure_planned_time"].max()),
}

# Hour rounding sanity check (on non-null rows)
sample_check = ice_df[ice_df["departure_planned_time"].notna()].head(3)
rounding_examples = [
    {
        "departure_planned_time": str(row["departure_planned_time"]),
        "departure_planned_hour": str(row["departure_planned_hour"]),
        "departure_planned_hour_naive": str(row["departure_planned_hour_naive"]),
    }
    for _, row in sample_check.iterrows()
]

failed = [k for k, v in validations.items() if isinstance(v, dict) and v.get("pass") is False]
if failed:
    raise ValueError(f"Validation failed: {failed}")

# =============================================================================
# 7. SAVE STANDARDIZED DATA + REPORT
# =============================================================================
ice_df.to_parquet(OUTPUT_PARQUET, index=False)

dtypes_after = {col: str(ice_df[col].dtype) for col in TIMESTAMP_COLS}

standardize_report = {
    "metadata": {
        "notebook": "03_Data_Cleaning_and_Standardization",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_month": TARGET_MONTH,
        "input_parquet": str(INPUT_PARQUET),
        "output_parquet": str(OUTPUT_PARQUET),
        "timezone": TARGET_TZ,
    },
    "rows": {
        "before": rows_before,
        "after": len(ice_df),
        "unchanged": True,
    },
    "dtypes_before": dtypes_before,
    "dtypes_after": dtypes_after,
    "timestamp_null_counts": {
        col: int(ice_df[col].isna().sum()) for col in TIMESTAMP_COLS
    },
    "dst_issues_new_nats": dst_issues,
    "merge_key": {
        "column": "departure_planned_hour",
        "naive_column": "departure_planned_hour_naive",
        "null_count": merge_key_nulls,
        "null_pct": merge_key_null_pct,
        "unique_hours": int(ice_df["departure_planned_hour_naive"].nunique()),
        "rounding_examples": rounding_examples,
    },
    "validations": validations,
    "columns_added": ["departure_planned_hour", "departure_planned_hour_naive"],
    "next_section": "Section 2 — exclude cancellations, handle nulls, save ice_cleaned",
}

with STANDARDIZE_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(standardize_report, f, indent=2, ensure_ascii=False)

# =============================================================================
# 8. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print("Section 1: Timestamp Standardization (Europe/Berlin)")
print(sep)
print(f"Rows (unchanged) : {len(ice_df):,}")
print(f"Timezone         : {TARGET_TZ}")
print()

print("--- DTYPE BEFORE → AFTER ---")
for col in TIMESTAMP_COLS:
    print(f"  {col}")
    print(f"    before: {dtypes_before[col]}")
    print(f"    after : {dtypes_after[col]}")
print()

print("--- NULL COUNTS (timestamp columns) ---")
for col in TIMESTAMP_COLS:
    nulls = ice_df[col].isna().sum()
    print(f"  {col:<28}: {nulls:>6,} null")
print()

print("--- HOUR ROUNDING EXAMPLES ---")
for ex in rounding_examples:
    print(f"  {ex['departure_planned_time']}  →  {ex['departure_planned_hour_naive']}")
print()

print("--- VALIDATIONS ---")
for key, val in validations.items():
    if "pass" in val:
        status = "PASS" if val["pass"] else "FAIL"
        print(f"  [{status}] {key}")
print()

if dst_issues:
    print(f"--- DST WARNINGS ---")
    for col, n in dst_issues.items():
        print(f"  {col}: {n} new NaT values from DST handling")
    print()

print(f"--- SAVED ---")
print(f"  Standardized data : {OUTPUT_PARQUET}")
print(f"  Report            : {STANDARDIZE_REPORT_PATH}")
print()
print("Later notebooks load with:")
print(f'  pd.read_parquet("{OUTPUT_PARQUET}")')
print()
print("Ready for Section 2: cancellation removal & null handling.")
print(sep)

Loading: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_raw_2024-07.parquet
Rows loaded: 157,886

  time                         → datetime64[ns, Europe/Berlin]  (null: 0)
  arrival_planned_time         → datetime64[ns, Europe/Berlin]  (null: 15,310)
  arrival_change_time          → datetime64[ns, Europe/Berlin]  (null: 15,307)
  departure_planned_time       → datetime64[ns, Europe/Berlin]  (null: 15,452)
  departure_change_time        → datetime64[ns, Europe/Berlin]  (null: 15,441)

Merge key created: departure_planned_hour
  Null merge keys : 15,452  (9.79%)
  Unique hours    : 744

Section 1: Timestamp Standardization (Europe/Berlin)
Rows (unchanged) : 157,886
Timezone         : Europe/Berlin

--- DTYPE BEFORE → AFTER ---
  time
    before: datetime64[ns]
    after : datetime64[ns, Europe/Berlin]
  arrival_planned_time
    before: datetime64[ns]
    after : datetime64[ns, Europe/Berlin]
  arrival_change_time
    before: datetime64[ns]
    after : datetime64[ns, E

# Section 2: Row Filtering, Null Handling & Cleaned Dataset Export

## Objective

Apply cleaning rules to the standardized ICE data: remove canceled stops, fix null station names, document null `line_number` values, validate the result, and save `ice_cleaned_2024-07.parquet`.

## Why this step is important

Canceled stops have no real delay outcome. Null station names break readability. Cleaning now prevents bad rows from entering the merge and models.

## What problem does it solve?

It answers: *Which rows should be excluded, how are nulls handled, and is the dataset modeling-ready?*

## Methodology

| Rule | Action | Reason |
|------|--------|--------|
| `is_canceled == True` | **Drop row** | No valid delay outcome |
| `station_name` is null | **Fill from `xml_station_name`** | Same station, better label |
| `line_number` is null | **Keep null** | Normal for ICE long-distance |
| Duplicate `id` | **Drop duplicates** | One row per stop |
| `train_type != ICE` | **Drop** | Safety check |

Save `ice_cleaned_2024-07.parquet` + `cleaning_report_2024-07.json`.

## Expected Output

- Row count before vs after cleaning
- Canceled rows removed count
- `station_name` nulls before vs after fill
- Validation checklist (all PASS)
- Saved cleaned parquet

## Interpretation

- **~1–5% rows dropped** → canceled stops removed (exact % from your data)
- **`line_number` still mostly null** → expected for ICE; not a data error
- **Row count drops only from cancellations/duplicates** → timestamps unchanged from Section 1
- **`ice_cleaned_2024-07.parquet`** → input for Notebook 04 merge

## Challenges Faced

| Challenge | Approach |
|-----------|----------|
| Canceled but has delay value | Exclude entirely — outcome is undefined |
| `station_name` null | Fill from `xml_station_name` |
| Null `departure_planned_time` | Keep row; flagged for merge (can't join weather) |

## Summary

Cleaned ICE dataset saved. Cancellations removed, station names fixed, validations passed.

## Next Step

**Section 3:** Notebook 03 close-out — verify all files, summarize cleaning, confirm readiness for Notebook 04 (geocoding + merge).

---

**Key Takeaways**
- Canceled stops excluded
- `station_name` filled from `xml_station_name`
- Output: `ice_cleaned_2024-07.parquet`

**What Comes Next**
- Notebook 04: geocode 95 `eva` stations + merge weather

In [3]:
# =============================================================================
# Notebook 03 | Section 2: Row Filtering, Null Handling & Cleaned Dataset Export
# =============================================================================
# Self-contained: loads ice_standardized parquet from disk.
# Saves ice_cleaned_2024-07.parquet for Notebook 04.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# =============================================================================
# 1. PATHS
# =============================================================================
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CONFIG_PATH = REFERENCE_DIR / "project_config.json"
MERGE_STRATEGY_PATH = REFERENCE_DIR / "merge_strategy.json"

TARGET_MONTH = "2024-07"
INPUT_PARQUET = PROCESSED_DIR / f"ice_standardized_{TARGET_MONTH}.parquet"
OUTPUT_PARQUET = PROCESSED_DIR / f"ice_cleaned_{TARGET_MONTH}.parquet"
CLEANING_REPORT_PATH = REFERENCE_DIR / f"cleaning_report_{TARGET_MONTH}.json"

ICE_FILTER = "ICE"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    """Convert numpy/pandas types to native Python for JSON serialization."""
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


# =============================================================================
# 2. LOAD DATA
# =============================================================================
config = load_json(CONFIG_PATH)
merge_strategy = load_json(MERGE_STRATEGY_PATH)

if not INPUT_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing: {INPUT_PARQUET}\nRun Notebook 03 Section 1 first."
    )

print(f"Loading: {INPUT_PARQUET}")
df = pd.read_parquet(INPUT_PARQUET)
rows_input = len(df)
print(f"Rows loaded: {rows_input:,}")
print()

# Track every cleaning step for the report
cleaning_log: list[dict] = []


def log_step(step: str, rows_before: int, rows_after: int, detail: str = "") -> None:
    removed = rows_before - rows_after
    cleaning_log.append({
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": removed,
        "detail": detail,
    })
    print(
        f"  [{step}] {rows_before:,} → {rows_after:,}  "
        f"(removed: {removed:,})  {detail}"
    )


# =============================================================================
# 3. CLEANING RULE 1 — keep ICE only (safety check)
# =============================================================================
print("--- CLEANING STEPS ---")
rows = len(df)
df = df[df["train_type"] == ICE_FILTER].copy()
log_step("filter_ICE_only", rows, len(df))

# =============================================================================
# 4. CLEANING RULE 2 — remove canceled stops
# =============================================================================
rows = len(df)
canceled_count = int(df["is_canceled"].sum())
df = df[df["is_canceled"] == False].copy()  # noqa: E712
log_step(
    "exclude_canceled",
    rows,
    len(df),
    detail=f"{canceled_count:,} canceled stops removed",
)

# =============================================================================
# 5. CLEANING RULE 3 — drop duplicate ids
# =============================================================================
rows = len(df)
dup_count = int(df["id"].duplicated().sum())
df = df.drop_duplicates(subset=["id"], keep="first")
log_step(
    "drop_duplicate_ids",
    rows,
    len(df),
    detail=f"{dup_count:,} duplicate ids removed",
)

# =============================================================================
# 6. CLEANING RULE 4 — fill null station_name from xml_station_name
# =============================================================================
station_name_nulls_before = int(df["station_name"].isna().sum())
df["station_name"] = df["station_name"].fillna(df["xml_station_name"])
station_name_nulls_after = int(df["station_name"].isna().sum())

cleaning_log.append({
    "step": "fill_station_name",
    "nulls_before": station_name_nulls_before,
    "nulls_after": station_name_nulls_after,
    "detail": "Filled from xml_station_name where station_name was null",
})
print(
    f"  [fill_station_name] nulls: {station_name_nulls_before:,} → "
    f"{station_name_nulls_after:,}"
)

# =============================================================================
# 7. CLEANING RULE 5 — document line_number nulls (keep, do not drop)
# =============================================================================
line_number_null_pct = round(100 * df["line_number"].isna().mean(), 2)
cleaning_log.append({
    "step": "line_number_nulls_kept",
    "null_pct": line_number_null_pct,
    "detail": "ICE rows often have null line_number — expected, rows kept",
})
print(f"  [line_number] null %: {line_number_null_pct}% — kept as-is (normal for ICE)")

# =============================================================================
# 8. FLAG ROWS THAT CANNOT BE MERGED (no departure_planned_time)
# =============================================================================
df["mergeable"] = df["departure_planned_time"].notna()
mergeable_count = int(df["mergeable"].sum())
non_mergeable_count = int((~df["mergeable"]).sum())
non_mergeable_pct = round(100 * non_mergeable_count / len(df), 2)

print(
    f"  [mergeable flag] mergeable: {mergeable_count:,} | "
    f"non-mergeable (no departure time): {non_mergeable_count:,} ({non_mergeable_pct}%)"
)
print()

# =============================================================================
# 9. VALIDATION
# =============================================================================
validations = {
    "only_ICE": {
        "pass": bool((df["train_type"] == ICE_FILTER).all()),
        "non_ICE_count": int((df["train_type"] != ICE_FILTER).sum()),
    },
    "no_canceled": {
        "pass": bool(not df["is_canceled"].any()),
        "canceled_count": int(df["is_canceled"].sum()),
    },
    "no_duplicate_ids": {
        "pass": bool(not df["id"].duplicated().any()),
        "duplicate_count": int(df["id"].duplicated().sum()),
    },
    "station_name_nulls": {
        "pass": bool(int(df["station_name"].isna().sum()) == 0),
        "null_count": int(df["station_name"].isna().sum()),
    },
    "timestamps_tz_aware": {
        "pass": bool(str(df["time"].dtype) == "datetime64[ns, Europe/Berlin]"),
        "dtype": str(df["time"].dtype),
    },
    "merge_key_present": {
        "pass": bool("departure_planned_hour_naive" in df.columns),
    },
    "row_count_reduced": {
        "input_rows": rows_input,
        "output_rows": len(df),
        "rows_removed": rows_input - len(df),
        "removal_pct": round(100 * (rows_input - len(df)) / rows_input, 2),
    },
}

failed = [k for k, v in validations.items() if v.get("pass") is False]
if failed:
    raise ValueError(f"Validation FAILED: {failed}")

# =============================================================================
# 10. DELAY STATS ON CLEANED DATA
# =============================================================================
delay_stats = {
    "mean": round(float(df["delay_in_min"].mean()), 2),
    "median": float(df["delay_in_min"].median()),
    "min": int(df["delay_in_min"].min()),
    "max": int(df["delay_in_min"].max()),
    "on_time_pct": round(100 * (df["delay_in_min"] == 0).mean(), 2),
    "delayed_gte_6_pct": round(100 * (df["delay_in_min"] >= 6).mean(), 2),
}

# =============================================================================
# 11. SAVE CLEANED DATA + REPORT
# =============================================================================
df.to_parquet(OUTPUT_PARQUET, index=False)

cleaning_report = {
    "metadata": {
        "notebook": "03_Data_Cleaning_and_Standardization",
        "section": "Section 2",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_month": TARGET_MONTH,
        "input_parquet": str(INPUT_PARQUET),
        "output_parquet": str(OUTPUT_PARQUET),
    },
    "rows": {
        "input": rows_input,
        "output": len(df),
        "removed": rows_input - len(df),
        "removal_pct": validations["row_count_reduced"]["removal_pct"],
    },
    "cleaning_log": cleaning_log,
    "mergeable": {
        "mergeable_rows": mergeable_count,
        "non_mergeable_rows": non_mergeable_count,
        "non_mergeable_pct": non_mergeable_pct,
        "note": (
            "Non-mergeable rows have no departure_planned_time — "
            "excluded from weather join in Notebook 04"
        ),
    },
    "line_number_null_pct": line_number_null_pct,
    "delay_stats_cleaned": delay_stats,
    "validations": validations,
    "columns_in_output": list(df.columns),
    "ready_for_notebook_04": True,
}

with CLEANING_REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(cleaning_report), f, indent=2, ensure_ascii=False)

# =============================================================================
# 12. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print(sep)
print("Section 2: Row Filtering, Null Handling & Cleaned Dataset Export")
print(sep)
print(f"Input rows  : {rows_input:,}")
print(f"Output rows : {len(df):,}")
print(f"Removed     : {rows_input - len(df):,}  ({validations['row_count_reduced']['removal_pct']}%)")
print()

print("--- VALIDATIONS ---")
for key, val in validations.items():
    if "pass" in val:
        print(f"  [{'PASS' if val['pass'] else 'FAIL'}] {key}")
print()

print("--- DELAY STATS (cleaned data) ---")
for k, v in delay_stats.items():
    print(f"  {k}: {v}")
print()

print("--- MERGE READINESS ---")
print(f"  Mergeable rows     : {mergeable_count:,}")
print(f"  Non-mergeable rows : {non_mergeable_count:,}  ({non_mergeable_pct}%)")
print(f"  Unique eva stations: {df['eva'].nunique()}")
print()

print("--- SAVED ---")
print(f"  Cleaned data : {OUTPUT_PARQUET}")
print(f"  Report       : {CLEANING_REPORT_PATH}")
print()
print("Later notebooks load with:")
print(f'  pd.read_parquet("{OUTPUT_PARQUET}")')
print()
print("Ready for Section 3: Notebook 03 close-out.")
print(sep)

Loading: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_standardized_2024-07.parquet
Rows loaded: 157,886

--- CLEANING STEPS ---
  [filter_ICE_only] 157,886 → 157,886  (removed: 0)  
  [exclude_canceled] 157,886 → 146,389  (removed: 11,497)  11,497 canceled stops removed
  [drop_duplicate_ids] 146,389 → 146,389  (removed: 0)  0 duplicate ids removed
  [fill_station_name] nulls: 0 → 0
  [line_number] null %: 100.0% — kept as-is (normal for ICE)
  [mergeable flag] mergeable: 132,268 | non-mergeable (no departure time): 14,121 (9.65%)

Section 2: Row Filtering, Null Handling & Cleaned Dataset Export
Input rows  : 157,886
Output rows : 146,389
Removed     : 11,497  (7.28%)

--- VALIDATIONS ---
  [PASS] only_ICE
  [PASS] no_canceled
  [PASS] no_duplicate_ids
  [PASS] station_name_nulls
  [PASS] timestamps_tz_aware
  [PASS] merge_key_present

--- DELAY STATS (cleaned data) ---
  mean: 10.92
  median: 4.0
  min: -46
  max: 298
  on_time_pct: 18.35
  delayed_gte_6_pct: 43.

# Section 3: Cleaning Summary & Notebook 03 Close-Out

## Objective

Verify all Notebook 03 files exist, summarize the cleaning pipeline, and confirm readiness for Notebook 04 (geocoding + weather merge).

## Why this step is important

A close-out proves the cleaning phase is complete and reproducible — Notebook 04 loads `ice_cleaned_2024-07.parquet` directly from disk.

## What problem does it solve?

It answers: *What changed from raw to cleaned, and are we ready to merge weather?*

## Methodology

Load all Notebook 03 artifacts, cross-check row counts across reports, print pipeline summary, save `notebook_03_summary.json`.

## Expected Output

- File checklist (all OK)
- Before/after row counts
- Cleaning steps recap
- Ready for Notebook 04: YES

## Interpretation

- **157,886 → 146,389 rows** → 11,497 canceled stops removed
- **132,268 mergeable** → can join weather in Notebook 04
- **14,121 non-mergeable** → no departure time; skipped in merge
- **95 stations** → need geocoding in Notebook 04

## Challenges Faced

| Challenge | Resolution |
|-----------|------------|
| JSON save error (`numpy.bool_`) | `make_json_safe()` helper |
| 9.65% non-mergeable rows | Flagged; excluded from weather join |
| 100% null `line_number` | Kept — normal for ICE |

## Summary

Notebook 03 complete. Timestamps standardized, data cleaned, merge key ready.

## Next Step

**Notebook 04, Section 1:** Geocode all 95 `eva` station IDs to latitude/longitude.

---

### Notebook 03 — Closing

**Key Takeaways**
- 146,389 cleaned ICE stops (July 2024)
- Timestamps in `Europe/Berlin`; merge key `departure_planned_hour_naive` ready
- 132,268 rows ready for weather merge

**What Comes Next**
- Notebook 04: geocode stations → fetch weather per station → merge → validate

In [4]:
# =============================================================================
# Notebook 03 | Section 3: Cleaning Summary & Notebook 03 Close-Out
# =============================================================================
# Self-contained: verifies all Notebook 03 artifacts and saves summary JSON.
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# =============================================================================
# 1. PATHS
# =============================================================================
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TARGET_MONTH = "2024-07"
SUMMARY_PATH = REFERENCE_DIR / "notebook_03_summary.json"

ARTIFACTS = {
    "ice_raw": PROCESSED_DIR / f"ice_raw_{TARGET_MONTH}.parquet",
    "ice_standardized": PROCESSED_DIR / f"ice_standardized_{TARGET_MONTH}.parquet",
    "ice_cleaned": PROCESSED_DIR / f"ice_cleaned_{TARGET_MONTH}.parquet",
    "timestamp_report": REFERENCE_DIR / f"timestamp_standardize_report_{TARGET_MONTH}.json",
    "cleaning_report": REFERENCE_DIR / f"cleaning_report_{TARGET_MONTH}.json",
}


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def check_file(path: Path) -> dict:
    if path.exists():
        return {
            "status": "OK",
            "path": str(path),
            "size_mb": round(path.stat().st_size / (1024 * 1024), 2),
        }
    return {"status": "MISSING", "path": str(path), "size_mb": None}


# =============================================================================
# 2. VERIFY ALL FILES
# =============================================================================
print("--- NOTEBOOK 03 ARTIFACT CHECK ---")
artifact_status = {name: check_file(path) for name, path in ARTIFACTS.items()}
for name, info in artifact_status.items():
    size = f"({info['size_mb']} MB)" if info["size_mb"] is not None else ""
    print(f"  [{info['status']}] {name} {size}")

missing = [n for n, i in artifact_status.items() if i["status"] == "MISSING"]
if missing:
    raise FileNotFoundError(
        f"Missing Notebook 03 files: {missing}\n"
        "Complete Sections 1–2 first."
    )

# =============================================================================
# 3. LOAD REPORTS + CLEANED DATA
# =============================================================================
ts_report = load_json(ARTIFACTS["timestamp_report"])
clean_report = load_json(ARTIFACTS["cleaning_report"])
ice_cleaned = pd.read_parquet(ARTIFACTS["ice_cleaned"])

# =============================================================================
# 4. BUILD PIPELINE SUMMARY
# =============================================================================
pipeline_summary = {
    "metadata": {
        "notebook": "03_Data_Cleaning_and_Standardization",
        "section": "Section 3 — Close-Out",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_month": TARGET_MONTH,
        "status": "COMPLETE",
    },
    "pipeline_steps": [
        {
            "step": 1,
            "section": "Section 1",
            "action": "Timestamp standardization → Europe/Berlin",
            "rows_in": ts_report["rows"]["before"],
            "rows_out": ts_report["rows"]["after"],
            "output_file": str(ARTIFACTS["ice_standardized"]),
        },
        {
            "step": 2,
            "section": "Section 2",
            "action": "Remove canceled, fill station_name, flag mergeable",
            "rows_in": clean_report["rows"]["input"],
            "rows_out": clean_report["rows"]["output"],
            "rows_removed": clean_report["rows"]["removed"],
            "output_file": str(ARTIFACTS["ice_cleaned"]),
        },
    ],
    "cleaned_dataset": {
        "rows": int(len(ice_cleaned)),
        "columns": int(len(ice_cleaned.columns)),
        "unique_eva_stations": int(ice_cleaned["eva"].nunique()),
        "mergeable_rows": clean_report["mergeable"]["mergeable_rows"],
        "non_mergeable_rows": clean_report["mergeable"]["non_mergeable_rows"],
        "non_mergeable_pct": clean_report["mergeable"]["non_mergeable_pct"],
        "delay_mean_min": clean_report["delay_stats_cleaned"]["mean"],
        "delayed_gte_6_pct": clean_report["delay_stats_cleaned"]["delayed_gte_6_pct"],
        "timezone": ts_report["metadata"]["timezone"],
        "merge_key": "departure_planned_hour_naive",
        "parquet_file": str(ARTIFACTS["ice_cleaned"]),
    },
    "artifacts": artifact_status,
    "ready_for_notebook_04": True,
    "notebook_04_tasks": [
        "Geocode 95 eva stations → latitude/longitude",
        "Fetch hourly weather per station for July 2024",
        "Left join weather on lat/lon + departure_planned_hour_naive",
        "Run merge validation checks from merge_strategy.json",
        "Save ice_weather_merged_2024-07.parquet",
    ],
}

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(pipeline_summary), f, indent=2, ensure_ascii=False)

# =============================================================================
# 5. PRINT SUMMARY
# =============================================================================
sep = "=" * 72
print()
print(sep)
print("Notebook 03 — Cleaning Summary")
print(sep)

raw_rows = ts_report["rows"]["before"]
std_rows = ts_report["rows"]["after"]
clean_rows = clean_report["rows"]["output"]
removed = clean_report["rows"]["removed"]

print()
print("--- PIPELINE ---")
print(f"  ice_raw_2024-07.parquet         : {raw_rows:>8,} rows")
print(f"  ice_standardized_2024-07.parquet: {std_rows:>8,} rows  (timestamps fixed)")
print(f"  ice_cleaned_2024-07.parquet     : {clean_rows:>8,} rows  ({removed:,} removed)")
print()

cd = pipeline_summary["cleaned_dataset"]
print("--- CLEANED DATASET ---")
print(f"  Rows              : {cd['rows']:,}")
print(f"  Stations (eva)    : {cd['unique_eva_stations']}")
print(f"  Mergeable         : {cd['mergeable_rows']:,}")
print(f"  Non-mergeable     : {cd['non_mergeable_rows']:,}  ({cd['non_mergeable_pct']}%)")
print(f"  Mean delay        : {cd['delay_mean_min']} min")
print(f"  Delayed ≥6 min    : {cd['delayed_gte_6_pct']}%")
print(f"  Timezone          : {cd['timezone']}")
print(f"  Merge key         : {cd['merge_key']}")
print()

print("--- SAVED FILES ---")
for name, info in artifact_status.items():
    print(f"  {info['path']}  ({info['size_mb']} MB)")
print(f"  {SUMMARY_PATH}")
print()

print("=" * 72)
print("NOTEBOOK 03 COMPLETE — Ready for Notebook 04")
print("=" * 72)
print()
print("Key Takeaways:")
print(f"  • {clean_rows:,} cleaned ICE stops | {cd['unique_eva_stations']} stations")
print(f"  • {cd['mergeable_rows']:,} rows ready for weather merge")
print("  • Timestamps standardized | merge key created")
print()
print("What Comes Next:")
print("  → Notebook 04, Section 1: Geocode eva stations to lat/lon")
print(sep)

--- NOTEBOOK 03 ARTIFACT CHECK ---
  [OK] ice_raw (5.24 MB)
  [OK] ice_standardized (5.4 MB)
  [OK] ice_cleaned (5.2 MB)
  [OK] timestamp_report (0.0 MB)
  [OK] cleaning_report (0.0 MB)

Notebook 03 — Cleaning Summary

--- PIPELINE ---
  ice_raw_2024-07.parquet         :  157,886 rows
  ice_standardized_2024-07.parquet:  157,886 rows  (timestamps fixed)
  ice_cleaned_2024-07.parquet     :  146,389 rows  (11,497 removed)

--- CLEANED DATASET ---
  Rows              : 146,389
  Stations (eva)    : 95
  Mergeable         : 132,268
  Non-mergeable     : 14,121  (9.65%)
  Mean delay        : 10.92 min
  Delayed ≥6 min    : 43.38%
  Timezone          : Europe/Berlin
  Merge key         : departure_planned_hour_naive

--- SAVED FILES ---
  c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_raw_2024-07.parquet  (5.24 MB)
  c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\processed\ice_standardized_2024-07.parquet  (5.4 MB)
  c:\Users\Manikanta\Desktop\ML Project\Notebooks\d